In [ ]:
import os
import pandas as pd

In [ ]:
%%writefile constants.py
EMBDEDDING_MODEL_PATH = "/kaggle/input/qwen-3-embedding/transformers/0.6b/1"
DATA_PATH = "/kaggle/input/jigsaw-agile-community-rules"

# https://huggingface.co/Qwen/Qwen3-Embedding-0.6B/blob/main/config_sentence_transformers.json
EMBEDDING_MODEL_QUERY = "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:"

CLEAN_TEXT = True
TOP_K = 1000//2
BATCH_SIZE = 128

In [ ]:
%%writefile utils.py
import pandas as pd
import torch.distributed as dist

from datasets import Dataset
from cleantext import clean
from tqdm.auto import tqdm

from constants import CLEAN_TEXT


def build_prompt(row):
    return f"""r/{row["subreddit"]}\nComment: {row["body"]}"""


def cleaner(text):
    return clean(
        text,
        fix_unicode=True,
        to_ascii=True,
        lower=False,
        no_line_breaks=False,
        no_urls=True,
        no_emails=True,
        no_phone_numbers=True,
        no_numbers=False,
        no_digits=False,
        no_currency_symbols=False,
        no_punct=False,
        replace_with_url="<URL>",
        replace_with_email="<EMAIL>",
        replace_with_phone_number="<PHONE>",
        lang="en",
    )



def get_dataframe_to_train(data_path):
    train_dataset = pd.read_csv(f"{data_path}/train.csv")
    test_dataset = pd.read_csv(f"{data_path}/test.csv")

    flatten = []
    flatten.append(train_dataset[["body", "rule", "subreddit", "rule_violation"]])
    
    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            sub_dataset = test_dataset[[f"{violation_type}_example_{i}", "rule", "subreddit"]].copy()
            sub_dataset = sub_dataset.rename(columns={f"{violation_type}_example_{i}": "body"})
            sub_dataset["rule_violation"] = 1 if violation_type == "positive" else 0
            flatten.append(sub_dataset)

    dataframe = pd.concat(flatten, axis=0)    
    dataframe = dataframe.drop_duplicates(ignore_index=True)
    return dataframe


def prepare_dataframe(dataframe):
    dataframe["prompt"] = dataframe.apply(build_prompt, axis=1)

    if CLEAN_TEXT:
        tqdm.pandas(desc="cleaner")
        dataframe["prompt"] = dataframe["prompt"].progress_apply(cleaner)

    if "rule_violation" in dataframe.columns:
        dataframe["rule_violation"] = dataframe["rule_violation"].map(
            {
                1: 1,
                0: -1,
            }
        )

    return dataframe

In [ ]:
%%writefile semantic.py
import pandas as pd

from sentence_transformers import SentenceTransformer
from sentence_transformers.util import semantic_search, dot_score
from tqdm.auto import tqdm

from utils import get_dataframe_to_train, prepare_dataframe
from constants import DATA_PATH, EMBDEDDING_MODEL_PATH, EMBEDDING_MODEL_QUERY, TOP_K, BATCH_SIZE



def get_scores(test_dataframe):
    corpus_dataframe = get_dataframe_to_train(DATA_PATH)
    corpus_dataframe = prepare_dataframe(corpus_dataframe)
    
    embedding_model = SentenceTransformer(
        model_name_or_path=EMBDEDDING_MODEL_PATH,
        device="cuda",
    )

    result = []
    for rule in tqdm(test_dataframe["rule"].unique(), desc=f"Generate scores for each rule"):
        test_dataframe_part = test_dataframe.query("rule == @rule").reset_index(drop=True)
        corpus_dataframe_part = corpus_dataframe.query("rule == @rule").reset_index(drop=True)
        corpus_dataframe_part = corpus_dataframe_part.reset_index(names="row_id")
        
        query_embeddings = embedding_model.encode(
            sentences=test_dataframe_part["prompt"].tolist(),
            prompt=EMBEDDING_MODEL_QUERY,
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_tensor=True,
            device="cuda",
            normalize_embeddings=True,
        )
        document_embeddings = embedding_model.encode(
            sentences=corpus_dataframe_part["prompt"].tolist(),
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_tensor=True,
            device="cuda",
            normalize_embeddings=True,
        )
        test_dataframe_part["semantic"] = semantic_search(
            query_embeddings,
            document_embeddings,
            top_k=TOP_K,
            score_function=dot_score,
        )
        def get_score(semantic):
            semantic = pd.DataFrame(semantic)
            semantic = semantic.merge(
                corpus_dataframe_part[["row_id", "rule_violation"]],
                how="left",
                left_on="corpus_id",
                right_on="row_id",
            )
            semantic["score"] = semantic["score"]*semantic["rule_violation"]
            return semantic["score"].sum()
            
        tqdm.pandas(desc=f"Add label for {rule=}")
        test_dataframe_part["rule_violation"] = test_dataframe_part["semantic"].progress_apply(get_score)
        result.append(test_dataframe_part[["row_id", "rule_violation"]].copy())
        
    submission = pd.concat(result, axis=0)
    return submission


def generate_submission(train=None): 
    test_dataframe = pd.read_csv(f"{DATA_PATH}/test.csv" if train is None else train)
    test_dataframe = prepare_dataframe(test_dataframe)
    
    submission = get_scores(test_dataframe)
    submission = test_dataframe[["row_id"]].merge(submission, on="row_id", how="left")
    submission.to_csv("semantic_submission.csv" if train is None else f'semantic_submission_train.csv', index=False)


if __name__ == "__main__":
    generate_submission()

In [ ]:
from semantic import generate_submission
# generate_submission()#XXX
generate_submission('train_aug.csv')

## DEBERTA START

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from tqdm import tqdm

In [ ]:
# Load and preprocess data
train_path = "/kaggle/input/jigsaw-agile-community-rules/train.csv"
test_path = "/kaggle/input/jigsaw-agile-community-rules/test.csv"
sample_sub_path = "/kaggle/input/jigsaw-agile-community-rules/sample_submission.csv"

df = pd.read_csv(train_path)
df["text"] = df["rule"] + " [SEP] " + df["body"]
df["label"] = df["rule_violation"].astype(float)

def add_data(dataframe):
    ret=[[],[]]
    for i in ['positive_example_1','positive_example_2','negative_example_1','negative_example_2']:
        tmp= (dataframe['rule']+' [SEP] '+ dataframe[i]).tolist()
        ret[0]+= tmp
        ret[1]+= [1]*len(tmp) if 'positive' in i else [0]*len(tmp)
    return ret

In [ ]:
test_df = pd.read_csv(test_path)

# Get augmented data from both df and test_df
augmented_train = add_data(df)
augmented_test = add_data(test_df)

# Combine original df with augmented data
augmented_texts =  df.text.tolist()+augmented_train[0] + augmented_test[0]
augmented_labels =  df.label.tolist()+augmented_train[1] + augmented_test[1]


# Create new augmented dataframe
augmented_df = pd.DataFrame({
    'text': augmented_texts,
    'label': augmented_labels
})
print(f'Before:{augmented_df.shape}')
augmented_df = augmented_df.groupby(augmented_df['text'].str.lower(), as_index=False).agg({
    'text': 'first',  # Keep the original case of the first occurrence
    'label': 'mean'   # Take mean of labels
})
print('After:',augmented_df.shape)
augmented_df['rule']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[0])
augmented_df['body']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[1])

rule_map= {i:j for j,i in enumerate(augmented_df.rule.str.lower().unique())}
augmented_df['rule_id']= augmented_df.rule.str.lower().map(rule_map)

augmented_df.head()

In [ ]:
augmented_df.to_csv('train_aug.csv',index=False)

In [ ]:
SEED = 42
NFOLDS = 5
MAX_LEN = 256//2
BATCH_SIZE = 16*2
EPOCHS = 5*2
MODEL_PATH = "/kaggle/input/deberta-v3-base/transformers/default/1/deberta-v3-base"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Set seeds
import random
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# Load tokenizer locally
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast = False)

In [ ]:
class JigsawDataset(Dataset):
    def __init__(self, texts, labels,rule_ids, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.rule_ids = rule_ids

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = self.tokenizer(
            text, padding='max_length', truncation=True, max_length=self.max_len, return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        item['rule_ids']= torch.tensor(self.rule_ids[idx])
        return item

class JigsawModel(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.base = AutoModel.from_pretrained(model_path)
        self.drop = nn.Dropout(0.15)
        self.out = nn.Linear(self.base.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.base(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        return self.out(self.drop(pooled)).squeeze(1)

def train_one_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    for batch in tqdm(loader):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        logits = model(input_ids, mask)
        loss = nn.BCEWithLogitsLoss()(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step()
        if scheduler:
            scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [ ]:
def validate(model, loader):
    model.eval()
    preds, targets, rule_ids_list = [], [], []
    total_loss = 0
    criterion = nn.BCEWithLogitsLoss()
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            rule_ids = batch["rule_ids"]  # Assuming this is already on CPU as integers
            
            logits = model(input_ids, mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            targets.extend(labels.cpu().numpy())
            rule_ids_list.extend(rule_ids.cpu().numpy() if torch.is_tensor(rule_ids) else rule_ids)
    
    # Convert to numpy arrays
    preds = np.array(preds)
    targets = np.array(targets)
    rule_ids_array = np.array(rule_ids_list)
    
    # Compute AUC per rule
    unique_rules = np.unique(rule_ids_array)
    rule_aucs = {}
    
    for rule_id in unique_rules:
        rule_mask = rule_ids_array == rule_id
        rule_preds = preds[rule_mask]
        rule_targets = targets[rule_mask]
        
        # Only compute AUC if we have both positive and negative samples for this rule
        if len(np.unique(rule_targets >= 0.5)) > 1:
            rule_auc = roc_auc_score(rule_targets >= 0.5, rule_preds)
            rule_aucs[rule_id] = rule_auc
        else:
            # If only one class present, we can't compute AUC
            rule_aucs[rule_id] = np.nan
    
    # Compute average AUC across rules (excluding NaN values)
    valid_aucs = [auc for auc in rule_aucs.values() if not np.isnan(auc)]
    avg_auc_per_rule = np.mean(valid_aucs) if valid_aucs else 0
    
    val_loss = total_loss / len(loader)
    # print(rule_aucs,'Rule_AUC')
    return avg_auc_per_rule, val_loss, preds

In [ ]:
from transformers import get_linear_schedule_with_warmup

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    folds = StratifiedKFold(n_splits=NFOLDS, shuffle=True, random_state=SEED)
    for fold, (tr_idx, val_idx) in enumerate(folds.split(augmented_df, augmented_df["rule"])):
        print('--------- ','FOLD: ',fold,' --------')
        all_preds = []
        val_ds = JigsawDataset(
            augmented_df.iloc[val_idx]['text'].tolist(), 
            augmented_df.iloc[val_idx]['label'].tolist(), 
            augmented_df.iloc[val_idx]['rule_id'].tolist(), 

            tokenizer, MAX_LEN
        )
    
        train_ds = JigsawDataset(
            augmented_df.iloc[tr_idx]['text'].tolist(), 
            augmented_df.iloc[tr_idx]['label'].tolist(), 
            augmented_df.iloc[tr_idx]['rule_id'].tolist(), 
            
            tokenizer, MAX_LEN,
        )
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    
        # Initialize classification model and load MLM pre-trained weights
        model = JigsawModel(MODEL_PATH).to(DEVICE)
        for name, param in model.named_parameters():
            if name.startswith('base.embedding'):
                param.requires_grad = False
                # print(name)
                
        print('Trainable Params: ',sum(i.numel() for i in model.parameters() if i.requires_grad))
       
        optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5,eps=1e-6)
        total_steps= EPOCHS*len(train_loader)
        warmup_steps= 0.1*total_steps
        scheduler = get_linear_schedule_with_warmup(
                optimizer,
                num_warmup_steps=warmup_steps,
                num_training_steps=total_steps,
            )

    
        best_auc=0
        for epoch in range(EPOCHS):
            print(f"Epoch {epoch+1}/{EPOCHS}")
            loss = train_one_epoch(model, train_loader, optimizer, scheduler)
            val_auc, val_loss ,val_preds = validate(model, val_loader)
            
                        
            print(f"Loss: {loss:.4f}, Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}")
            if val_auc > best_auc:
                best_auc = val_auc
                torch.save(model.state_dict(), f"model_fold{fold}_auc.bin")
    
        all_preds.append(pd.Series(val_preds))

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    sample = pd.read_csv(sample_sub_path)
    df_test = pd.read_csv(test_path)
    df_test["text"] = df_test["rule"] + " [SEP] " + df_test["body"]
    
    test_preds = []
    for fold in range(NFOLDS):
        model = JigsawModel(MODEL_PATH).to(DEVICE)
        wts=torch.load(f"model_fold{fold}_auc.bin", map_location=DEVICE)
        wts= {k.replace('module.',''):v for k,v in wts.items()}
            
        model.load_state_dict(wts)
        model.eval()
    
        test_ds = JigsawDataset(df_test['text'].tolist(), [0]*len(df_test),[0]*len(df_test), tokenizer, MAX_LEN)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)
    
        fold_preds = []
        with torch.no_grad():
            for batch in test_loader:
                ids = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                logits = model(ids, mask)
                fold_preds.extend(torch.sigmoid(logits).cpu().numpy())
        test_preds.append(fold_preds)

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    final_preds = np.mean(test_preds, axis=0)
    sample["rule_violation"] = final_preds
    sample.to_csv("submission.csv", index=False)
    print("✅ Submission saved as submission.csv")
else:
    !touch submission.csv
    
!head -n 4 submission.csv

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # STAGE 1: Train models for pseudo labeling
    print("=== STAGE 1: Training models for pseudo labeling ===")
    
    stage1_models = []
    oof=[]
    val_id=[]
    folds = StratifiedKFold(n_splits=NFOLDS, shuffle=True, random_state=SEED)
    
    for fold, (tr_idx, val_idx) in enumerate(folds.split(augmented_df, augmented_df["rule"])):
        print(f'--------- FOLD {fold} (Stage 1) --------')
        torch.cuda.empty_cache()
        gc.collect()
        
        val_ds = JigsawDataset(
            augmented_df.iloc[val_idx]['text'].tolist(), 
            augmented_df.iloc[val_idx]['label'].tolist(), 
            augmented_df.iloc[val_idx]['rule_id'].tolist(), 
            tokenizer, MAX_LEN
        )
    
        train_ds = JigsawDataset(
            augmented_df.iloc[tr_idx]['text'].tolist(), 
            augmented_df.iloc[tr_idx]['label'].tolist(), 
            augmented_df.iloc[tr_idx]['rule_id'].tolist(), 
            tokenizer, MAX_LEN
        )
        
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

        model = JigsawModel(MODEL_PATH2).to(DEVICE)
        for name, param in model.named_parameters():
            if name.startswith('base.embedding'):
                param.requires_grad = False
                # print(name)
        model.load_state_dict(torch.load(f"model_fold{fold}_stage1.bin", map_location=DEVICE))
        print('Trainable Params: ',sum(i.numel() for i in model.parameters() if i.requires_grad))
        optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, eps=1e-6)
        total_steps = EPOCHS * len(train_loader)
        warmup_steps = int(0.1 * total_steps)
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps
        )
    
        # best_auc = 0
        # for epoch in range(EPOCHS):
        #     print(f"Epoch {epoch+1}/{EPOCHS}")
        #     loss = train_one_epoch(model, train_loader, optimizer, scheduler)
        val_auc, val_loss, val_preds,targets = validate(model, val_loader) #XXX
            
        #     print(f"Loss: {loss:.4f}, Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}")
        #     if val_auc > best_auc:
        #         best_auc = val_auc
        #         model_to_save = model.module if hasattr(model, 'module') else model
        #         torch.save(model_to_save.state_dict(), f"model_fold{fold}_stage1.bin")
        val_id.extend(list(val_idx))
        oof.append((val_preds,targets))
        stage1_models.append(f"model_fold{fold}_stage1.bin")
        del model, optimizer, scheduler, train_ds, val_ds, train_loader, val_loader

In [ ]:
sml=df[['rule','body']].copy()
sml['text']= sml["rule"] + " [SEP] " + sml["body"]
sml['preds']= oofllm[0][0].tolist()
sml=sml.groupby(sml.text.str.lower(),as_index=False).agg({
    'text': 'first',  # Keep the original case of the first occurrence
    'preds': 'mean'   # Take mean of labels
})
sml.shape

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from itertools import product

def blend_oofs_grid_search(oofs_dict, y_true, step=0.05, normalize=True):
    """
    Find optimal weights for blending multiple OOF predictions using grid search.
    
    Parameters:
    -----------
    oofs_dict : dict
        Dictionary with model names as keys and OOF predictions as values
        Example: {'model1': oof1_array, 'model2': oof2_array, ...}
    y_true : array-like
        True target values (must be aligned with OOF predictions)
    step : float
        Grid search step size for weights (default: 0.05 for 5% increments)
    normalize : bool
        Whether to normalize weights to sum to 1 (default: True)
    
    Returns:
    --------
    dict with keys:
        - 'best_weights': dictionary of optimal weights
        - 'best_auc': best AUC score achieved
        - 'best_blend': blended predictions using best weights
        - 'results_df': DataFrame with all tested combinations
    """
    
    model_names = list(oofs_dict.keys())
    n_models = len(model_names)
    
    print(f"Grid searching optimal weights for {n_models} models...")
    print(f"Step size: {step}")
    
    # Generate weight combinations
    if normalize:
        # Generate weights that sum to 1
        weights_range = np.arange(0, 1 + step, step)
        weight_combinations = []
        
        # Generate all combinations and filter those that sum to ~1
        for weights in product(weights_range, repeat=n_models):
            if abs(sum(weights) - 1.0) < 1e-6:  # Account for floating point errors
                weight_combinations.append(weights)
    else:
        # Generate independent weights
        weights_range = np.arange(0, 1 + step, step)
        weight_combinations = list(product(weights_range, repeat=n_models))
    
    print(f"Testing {len(weight_combinations)} weight combinations...")
    
    # Store results
    results = []
    best_auc = -1
    best_weights = None
    best_blend = None
    
    # Convert OOFs to numpy arrays
    oofs_arrays = [np.array(oofs_dict[name]) for name in model_names]
    y_true = np.array(y_true)
    
    # Test each combination
    for i, weights in enumerate(weight_combinations):
        # Blend predictions
        blended = np.zeros_like(oofs_arrays[0], dtype=float)
        for oof, w in zip(oofs_arrays, weights):
            blended += w * oof
        
        # Calculate AUC
        try:
            auc = roc_auc_score(y_true, blended)
            
            # Store result
            result = {name: w for name, w in zip(model_names, weights)}
            result['auc'] = auc
            results.append(result)
            
            # Update best
            if auc > best_auc:
                best_auc = auc
                best_weights = dict(zip(model_names, weights))
                best_blend = blended.copy()
        
        except Exception as e:
            print(f"Error with weights {weights}: {e}")
            continue
        
        # Progress update
        if (i + 1) % 1000 == 0:
            print(f"Processed {i + 1}/{len(weight_combinations)} combinations. Best AUC so far: {best_auc:.6f}")
    
    # Create results DataFrame
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('auc', ascending=False).reset_index(drop=True)
    
    print("\n" + "="*60)
    print("BEST COMBINATION FOUND:")
    print("="*60)
    for name, weight in best_weights.items():
        print(f"{name}: {weight:.3f}")
    print(f"\nBest AUC: {best_auc:.6f}")
    print("="*60)
    
    # Show top 10 combinations
    print("\nTop 10 Weight Combinations:")
    print(results_df.head(10).to_string(index=False))
    
    return {
        'best_weights': best_weights,
        'best_auc': best_auc,
        'best_blend': best_blend,
        'results_df': results_df
    }



# Example usage
if __name__ == "__main__":
    # Example: Generate synthetic data
    # np.random.seed(42)
    # n_samples = 1000
    
    # True labels
    y_true = all_targets

    oof1= bert_preds
    oof2= sml.preds
    
    # Create dictionary of OOFs
    oofs = {
        'deberta': oof1,
        'llm': oof2,
    }
    
    # Individual model AUCs
    print("Individual Model AUCs:")
    for name, oof in oofs.items():
        auc = roc_auc_score(y_true, oof)
        print(f"{name}: {auc:.6f}")
    print()
    
    # Run grid search
    results = blend_oofs_grid_search(
        oofs_dict=oofs,
        y_true=y_true,
        step=0.05,  # Use 0.05 or 0.025 for finer search
        normalize=True
    )
    
    # Access results
    best_weights = results['best_weights']
    best_auc = results['best_auc']
    best_blend = results['best_blend']
    results_df = results['results_df']